In [1]:
# Load master file 
import pandas as pd 
master = pd.read_parquet("D:\\instacart_market_analysis\\master_table.parquet")
master.head()

,order_id,product_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,...,department_id,aisle,department,time_of_order,day_type,order_frequency,order_stage,basket_size,order_reorder_count,order_reorder_rate
0,2,33120,1,1,202279,prior,3,5,9,8.0,...,16,Eggs,Dairy Eggs,Morning,Weekday,Bi Weekly,Growing,9,6,0.67
1,2,28985,2,1,202279,prior,3,5,9,8.0,...,4,Fresh Vegetables,Produce,Morning,Weekday,Bi Weekly,Growing,9,6,0.67
2,2,9327,3,0,202279,prior,3,5,9,8.0,...,13,Spices Seasonings,Pantry,Morning,Weekday,Bi Weekly,Growing,9,6,0.67
3,2,45918,4,1,202279,prior,3,5,9,8.0,...,13,Oils Vinegars,Pantry,Morning,Weekday,Bi Weekly,Growing,9,6,0.67
4,2,30035,5,0,202279,prior,3,5,9,8.0,...,13,Baking Ingredients,Pantry,Morning,Weekday,Bi Weekly,Growing,9,6,0.67


In [2]:
# Top 20 most sold products 
top_products = (master.groupby("product_name")["order_id"]
                .nunique()
                .reset_index()
                .rename(columns={"order_id":"total_orders"})
                )
top_products = top_products.sort_values(by = "total_orders",ascending = False)
print(top_products.head(20))               

                   product_name  total_orders
3648                     Banana        472565
3443     Bag Of Organic Bananas        379450
31870      Organic Strawberries        264683
28789      Organic Baby Spinach        241921
30247      Organic Hass Avocado        213584
28754           Organic Avocado        176815
22376               Large Lemon        152657
42838              Strawberries        142951
23381                     Limes        140627
32428        Organic Whole Milk        137905
31312       Organic Raspberries        137057
32515      Organic Yellow Onion        113426
29950            Organic Garlic        109778
32555          Organic Zucchini        104823
28957       Organic Blueberries        100060
11597            Cucumber Kirby         97315
29930        Organic Fuji Apple         89632
30527             Organic Lemon         87746
2625   Apple Honeycrisp Organic         85020
30089    Organic Grape Tomatoes         84255


In [3]:
# Top 20 most reordered products
reordered_products = (master[master["reordered"] == 1].groupby("product_name")["order_id"]
					  .nunique()
					  .reset_index()
					  .rename(columns={"order_id":"reorder_count"})
					  .sort_values(by="reorder_count", ascending=False)
					  )
print(reordered_products.head(20))

                   product_name  reorder_count
3221                     Banana         398609
3032     Bag Of Organic Bananas         315913
29034      Organic Strawberries         205845
26106      Organic Baby Spinach         186884
27487      Organic Hass Avocado         170131
26071           Organic Avocado         134044
29552        Organic Whole Milk         114510
20268               Large Lemon         106255
28504       Organic Raspberries         105409
39078              Strawberries          99802
21183                     Limes          95768
29631      Organic Yellow Onion          79072
27211            Organic Garlic          74663
29669          Organic Zucchini          72165
10476            Cucumber Kirby          67313
27191        Organic Fuji Apple          63811
26257       Organic Blueberries          62922
2300   Apple Honeycrisp Organic          62510
27751             Organic Lemon          60536
27465       Organic Half & Half          59672


In [4]:
# Top 20 products with highest reordered rate 
product_stats = (master.groupby(["product_id","product_name"]).agg(
                 total_orders =("order_id","nunique"),
                reorder_count = ("reordered","sum")
				)
					.reset_index() )
product_stats["reordered_rate"] = (product_stats["reorder_count"] / product_stats["total_orders"].round(4))

product_stats["reordered_rate_pct"] = (product_stats["reordered_rate"] * 100).round(2)

top_20_reordered_rate = (product_stats[product_stats["total_orders"] >= 100]
                        .sort_values(by = "reordered_rate", ascending = False)
                        .head(20))

top_20_reordered_rate = top_20_reordered_rate[["product_id","product_name","total_orders","reorder_count","reordered_rate","reordered_rate_pct"]]
print(top_20_reordered_rate)

       product_id                                       product_name  \
27734       27740                                 Chocolate Love Bar   
35598       35604                                    Maca Buttercups   
38243       38251                              Benchbreak Chardonnay   
10232       10236  Fragrance Free Clay With Natural Odor Eliminat...   
20594       20598                         Thousand Island Salad Snax   
35490       35496                       Real2 Alkalized Water 500 Ml   
9288         9292                    Half And Half Ultra Pasteurized   
45495       45504                         Whole Organic Omega 3 Milk   
43386       43394                    Organic Lactose Free Whole Milk   
5511         5514                     Organic Homogenized Whole Milk   
47220       47231                               Ultra-Purified Water   
45026       45035                               Coffee Flavor Yogurt   
17465       17469                               Lo-Carb Energy D

In [5]:
# Botom 10 product with lowest reorder rate
bottom_10_reorder_rate = (product_stats[product_stats["total_orders"]>=100]
                     .sort_values(by = "reordered_rate", ascending= True)
                     .head(10))


bottom_10_reorder_rate = bottom_10_reorder_rate[["product_name", "product_id","total_orders", "reorder_count","reordered_rate", "reordered_rate_pct"]]
print(bottom_10_reorder_rate)

                     product_name  product_id  total_orders  reorder_count  \
11668                 Ground Sage       11672           203              1   
10372       Organic Caraway Seeds       10376           134              1   
24360  Organic Chinese Five Spice       24364           169              2   
14290      Cornmeal, Stone Ground       14294           123              2   
13369               Food Coloring       13373           123              2   
29005                 Rubbed Sage       29011           122              2   
30025      Cajun Street Seasoning       30031           102              2   
32650                Garam Masala       32656           143              3   
6500        Ground Coriander Seed        6503           140              3   
48508            Green Food Color       48519           135              3   

       reordered_rate  reordered_rate_pct  
11668        0.004926                0.49  
10372        0.007463                0.75  
24360    

In [6]:
# organic vs non organic order count
master["is_organic"] = master["product_name"].str.contains("Organic", case=False, na=False).astype(int)

organic_orders = (master.groupby("is_organic")["order_id"]
                  .nunique()
                  .reset_index()
                  .rename(columns={"order_id": "total_orders"}))

organic_orders["label"] = organic_orders["is_organic"].map({1: "Organic", 0: "Non Organic"})
organic_orders = organic_orders[["label", "is_organic", "total_orders"]]
print(organic_orders)

         label  is_organic  total_orders
0  Non Organic           0       3135883
1      Organic           1       2367452


In [7]:
# organic vs non organic reorder rate
organic_reorder = (master.groupby("is_organic")["reordered"]
                   .mean()
                   .round(4)
                   .reset_index()
                   .rename(columns={"reordered": "reorder_rate"}))

organic_reorder["reorder_rate_pct"] = (organic_reorder["reorder_rate"] * 100).round(2)
organic_reorder["label"] = organic_reorder["is_organic"].map({1: "Organic", 0: "Non Organic"})
organic_reorder = organic_reorder[["label", "is_organic", "reorder_rate", "reorder_rate_pct"]]
print(organic_reorder)

         label  is_organic  reorder_rate  reorder_rate_pct
0  Non Organic           0        0.5688             56.88
1      Organic           1        0.6349             63.49


In [8]:
product_stats = product_stats.merge(
    master[["product_id","department","aisle","is_organic"]].drop_duplicates("product_id"),
    on="product_id", how="left"
)

In [9]:
product_stats["organic_label"] = product_stats["is_organic"].map({1: "Organic", 0: "Non Organic"})
product_stats.to_csv("D:\\instacart_market_analysis\\product_features.csv", index=False)

In [10]:
# aisle order count and reorder rate 
aisles_stats = (master.groupby(["aisle_id","aisle"])
                 .agg(
                    aisle_order_count = ("order_id","nunique"),
                    aisle_reorder_count = ("reordered","sum"),
                    total_items = ("product_id","count")
                    )
                    .reset_index())

aisles_stats["reorder_rate"] = (aisles_stats["aisle_reorder_count"] / aisles_stats["total_items"]).round(4)
aisles_stats["reorder_rate_pct"] = (aisles_stats["reorder_rate"] * 100).round(2)
aisles_stats = aisles_stats[["aisle","aisle_id","aisle_order_count","aisle_reorder_count","total_items","reorder_rate","reorder_rate_pct"]]
print(aisles_stats)


                          aisle  aisle_id  aisle_order_count  \
0         Prepared Soups Salads         1              63115   
1             Specialty Cheeses         2              77171   
2           Energy Granola Bars         3             278151   
3                 Instant Foods         4             165541   
4    Marinades Meat Preparation         5              58390   
..                          ...       ...                ...   
129    Hot Cereal Pancake Mixes       130             145294   
130                   Dry Pasta       131             227113   
131                      Beauty       132               5837   
132  Muscles Joints Pain Relief       133              17785   
133  Specialty Wines Champagnes       134               9885   

     aisle_reorder_count  total_items  reorder_rate  reorder_rate_pct  
0                  42912        71928        0.5966             59.66  
1                  40365        82491        0.4893             48.93  
2              

In [11]:
# top 15 aisle by total order count
top_15_aisle = (aisles_stats.sort_values("aisle_order_count",ascending= False).head(15))
print(top_15_aisle[["aisle","aisle_order_count"]])


                             aisle  aisle_order_count
23                    Fresh Fruits            1790771
82                Fresh Vegetables            1427631
122     Packaged Vegetables Fruits            1179243
119                         Yogurt             847081
83                            Milk             785987
20                 Packaged Cheese             737899
114  Water Seltzer Sparkling Water             614081
90                 Soy Lactosefree             545714
106                 Chips Pretzels             538052
111                          Bread             527129
85                            Eggs             440410
30                    Refrigerated             429510
115                 Frozen Produce             395743
77                        Crackers             368577
36                   Ice Cream Ice             352768


In [12]:
top_15_aisle_reorder = (aisles_stats[aisles_stats["aisle_order_count"] >= 1000]
                        .sort_values("reorder_rate", ascending=False)
                        .head(15))

print(top_15_aisle_reorder[["aisle", "aisle_order_count", "reorder_rate_pct"]])

                             aisle  aisle_order_count  reorder_rate_pct
83                            Milk             785987             78.14
114  Water Seltzer Sparkling Water             614081             72.96
23                    Fresh Fruits            1790771             71.81
85                            Eggs             440410             70.54
90                 Soy Lactosefree             545714             69.26
31                Packaged Produce             199356             69.07
119                         Yogurt             847081             68.65
52                           Cream             295032             68.50
111                          Bread             527129             67.02
30                    Refrigerated             429510             66.33
92                Breakfast Bakery             220871             65.12
63            Energy Sports Drinks              77618             64.96
76                     Soft Drinks             278132           

In [13]:
#aisle reorder rate vs order count
aisle_scatter = aisles_stats[["aisle","aisle_order_count","reorder_rate_pct"]]
print(aisle_scatter.sort_values("aisle_order_count",ascending= False).head(20))

                             aisle  aisle_order_count  reorder_rate_pct
23                    Fresh Fruits            1790771             71.81
82                Fresh Vegetables            1427631             59.45
122     Packaged Vegetables Fruits            1179243             63.85
119                         Yogurt             847081             68.65
83                            Milk             785987             78.14
20                 Packaged Cheese             737899             58.52
114  Water Seltzer Sparkling Water             614081             72.96
90                 Soy Lactosefree             545714             69.26
106                 Chips Pretzels             538052             58.88
111                          Bread             527129             67.02
85                            Eggs             440410             70.54
30                    Refrigerated             429510             66.33
115                 Frozen Produce             395743           

In [14]:

aisles_stats.to_csv("D:\\instacart_market_analysis\\aisle_features.csv", index=False)
print("Aisle features saved — shape:", aisles_stats.shape)

Aisle features saved — shape: (134, 7)


In [15]:
# department order count and reorder rate 
dept_stats = (master.groupby(["department_id","department"])
              .agg(
                  dept_order_count = ("order_id","nunique"),
                  dept_reorder_count = ("reordered","sum"),
                  total_item = ("product_id","count")
                  )
                .reset_index())
dept_stats["reorder_rate"] = (dept_stats["dept_reorder_count"]/dept_stats["total_item"]).round(4)
dept_stats["reorder_rate_pct"] = (dept_stats["reorder_rate"] * 100).round(2)
print(dept_stats)

    department_id       department  dept_order_count  dept_reorder_count  \
0               1           Frozen           1181018             1211890   
1               2            Other             35056               14806   
2               3           Bakery            881556              739188   
3               4          Produce           2409320             6160710   
4               5          Alcohol             84689               87595   
5               6    International            221537               99416   
6               7        Beverages           1457351             1757892   
7               8             Pets             59282               58760   
8               9  Dry Goods Pasta            597862              399581   
9              10             Bulk             33802               19950   
10             11    Personal Care            318555              143584   
11             12     Meat Seafood            574731              402442   
12          

In [16]:
# top orderd department 
top_dept = (dept_stats.sort_values("dept_order_count",ascending= False))
top_dept = top_dept[["department","dept_order_count"]]
print(top_dept)

         department  dept_order_count
3           Produce           2409320
15       Dairy Eggs           2177338
6         Beverages           1457351
18           Snacks           1391447
0            Frozen           1181018
12           Pantry           1117892
2            Bakery            881556
19             Deli            770300
14     Canned Goods            681305
8   Dry Goods Pasta            597862
11     Meat Seafood            574731
13        Breakfast            525188
16        Household            470780
10    Personal Care            318555
5     International            221537
17           Babies            177712
4           Alcohol             84689
20          Missing             59477
7              Pets             59282
1             Other             35056
9              Bulk             33802


In [17]:
# reorder rate per dept
top_dept_reorder = (dept_stats.sort_values("reorder_rate",ascending=False))
print(top_dept_reorder[["department","reorder_rate"]])

         department  reorder_rate
15       Dairy Eggs        0.6700
6         Beverages        0.6535
3           Produce        0.6499
2            Bakery        0.6281
19             Deli        0.6077
7              Pets        0.6013
17           Babies        0.5790
9              Bulk        0.5770
18           Snacks        0.5742
4           Alcohol        0.5699
11     Meat Seafood        0.5677
13        Breakfast        0.5609
0            Frozen        0.5419
8   Dry Goods Pasta        0.4611
14     Canned Goods        0.4574
1             Other        0.4080
16        Household        0.4022
20          Missing        0.3958
5     International        0.3692
12           Pantry        0.3467
10    Personal Care        0.3211


In [18]:
# average basket contribution per department
# how many items per order come from each department on average
dept_basket = (master.groupby(["order_id", "department"])["product_id"]
               .count()
               .reset_index()
               .rename(columns={"product_id": "items_from_dept"}))

dept_basket_avg = (dept_basket.groupby("department")["items_from_dept"]
                   .mean()
                   .round(2)
                   .reset_index()
                   .rename(columns={"items_from_dept": "avg_items_per_order"})
                   .sort_values("avg_items_per_order", ascending=False))

print(dept_basket_avg)

         department  avg_items_per_order
19          Produce                 3.93
7        Dairy Eggs                 2.49
1            Babies                 2.38
20           Snacks                 2.08
10           Frozen                 1.89
3         Beverages                 1.85
0           Alcohol                 1.81
16           Pantry                 1.68
18             Pets                 1.65
11        Household                 1.57
6      Canned Goods                 1.57
9   Dry Goods Pasta                 1.45
17    Personal Care                 1.40
8              Deli                 1.36
4         Breakfast                 1.35
2            Bakery                 1.33
13     Meat Seafood                 1.23
12    International                 1.22
14          Missing                 1.16
15            Other                 1.04
5              Bulk                 1.02


In [19]:
# save department features
dept_stats.to_csv("D:\\instacart_market_analysis\\dept_features.csv", index=False)
print("Department features saved — shape:", dept_stats.shape)

Department features saved — shape: (21, 7)
